<br> 

***  

# <center>**Notebook 08 – Training and Evaluation**</center>

<br>

<center>Supporting materials for</center>

### <center>MSc Data Science</center>
## <center>Module: DSM500 Final Project</center>

<br>

<center>5 October 2026</center>

***
<br>

**Purpose of this notebook**  
This notebooks implements XGBoost and Gaussian naive Bayes models using all unaugmented and augmented scenarios. Models are evaluated first using k-fold cross-validation (CV) and then using the common test set. Threshold tuning is performed for each CV fold using the unaugmented validation fold, and the median of the best fold thresholds is used for test set evaluation.  

Models are implemented as four primary functions each supported by helper functions. These can either be called directly, or from corresponding wrapper functions which iterate through folds and augmentation levels. Separate functions are implemented to aggregate fold and scenario-level results and to calculate the normalised F1 recovery metric. Although models are not saved because of their large number, raw evaluation results, probability predictions, and calculated metrics are saved.

XGBoost functions:  
* `hf_train_and_evaluate_fold` - Cross validation training for one fold  
* `hf_train_and_evaluate_scenario` - Evaluation against the test set  
* `hf_fold_training` - Wrapper function that calls `hf_train_and_evaluate_fold` for multiple folds and augmentation levels  
* `hf_scenario_training` - Wrapper function that calls `hf_train_and_evaluate_scenario` for multiple augmentation levels  

Gaussian naive Bayes functions:  
* `train_and_evaluate_gnb_fold` -  Cross validation training for one fold  
* `train_and_evaluate_gnb_scenario` - Evaluation against the test set  
* `hf_fold_training_gnb` - Wrapper function that calls `hf_train_and_evaluate_fold` for multiple folds and augmentation levels  
* `hf_scenario_training_gnb` - Wrapper function that calls `hf_train_and_evaluate_scenario` for multiple augmentation levels

Results aggregation and F1 recovery calculation:  
* `f1_recovery_fold` - Aggregate fold-level results and calculate F1 recovery
* `f1_recovery_scenario` - Aggregate scenario-level results and calculate F1 recovery
* `hf_save_f1_recovery_model` - Call `f1_recovery_fold` and `f_recovery_scenario` for XGBoost or Gaussian naive Bayes and save the results
 ```
..
    |-- scenarios
        |-- <size tier>-ir<IR>-b<bootstrap> e.g. large-ir01-b01
            |-- folds
            |-- synthetic
            |-- fold-results
                |-- results-fold<n>-aug<a>.json
                |-- threshold-fold<n>-aug<a>.npy
                |-- y-proba-fold<n>-aug<a>.npy
                |-- y-true-fold<n>-aug<a>.npy
            |-- scenrio-results
                |-- results-aug<a>.json
                |-- threshold-aug<a>.npy
                |-- y-proba-aug<a>.npy
                |-- y-true-aug<a>.npy
    |-- results
        |-- xgb-fold-results-b00.json         # XGB fold evaluation results for all scenarios 
        |-- xgb-scenarios-results-b00.json    # XGB scenario evaluation results for all scenarios
        |-- gnb-fold-results-b00.json         # GNB fold evaluation results for all scenarios
        |-- gnb-scenario-results-b00.json     # GNB scenario evaluation results for all scenarios
```
Where: 
* `n` is fold number
* `a` is augmentation level percentage


## Load the training and test sets

In [3]:
import pandas as pd

# Read the training set. Convert the label DataFrames back to Series
X_train = pd.read_parquet('..\\data\\processed\\X_train.parquet', engine='pyarrow')
y_train = pd.read_parquet('..\\data\\processed\\y_train.parquet', engine='pyarrow')['TARGET']
X_test = pd.read_parquet('..\\data\\processed\\X_test.parquet', engine='pyarrow')
y_test = pd.read_parquet('..\\data\\processed\\y_test.parquet', engine='pyarrow')['TARGET']

# Confirm the data load
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(276759, 125) (276759,) (30752, 125) (30752,)


In [4]:
# TEST
if False:
    print(X_train.head(2))
    print(y_train.head(2))
    print(X_test.head(2))
    print(y_test.head(2))

## Training and evaluation helper functions

This section implements the evaluation functions, and helper functions used by XGBoost and GaussianNB.

### Read real data for the fold or scenario (`hf_read_real_data`)

In [5]:
from pathlib import Path
import numpy as np

def hf_read_real_data(tier, ir, bootstrap, fold):
    '''Get the training and evaluation data for a scenario or CV fold. fold=-1 indicates scenario.
    
    Summary
    -------
    Generate the path to the fold indices from the input parameters. Read the indices then select
    the training and eval fold data from the full training set and return the results.
    '''
    # Get the path to the indices
    scenario = f'{tier}-ir{ir:03d}-b{bootstrap:02}'
    root = Path('..') / 'scenarios' / scenario

    # Read scenario-level or fold-level indices depending on the fold value, and return a slice 
    # of the training set
    if fold == -1:
        # Scenario-level data
        train_file = f'training-{scenario}.txt'
        train_path = root / train_file
        train_indices = np.loadtxt(train_path, dtype=int)              # Read the scenario indices

        return  X_train.loc[train_indices], y_train.loc[train_indices] # Return the DataFrames
    else:
        # Fold-level data
        train_file = f'fold-{fold:1}-train-ids.txt'
        eval_file = f'fold-{fold:1}-eval-ids.txt'
        train_fold_path = root / 'folds' / train_file
        eval_fold_path = root / 'folds' / eval_file

        train_fold_indices = np.loadtxt(train_fold_path, dtype=int)   # Read the train indices
        eval_fold_indices = np.loadtxt(eval_fold_path, dtype=int)     # Read the val indices

        return (                                                      # Return the DataFrames
            X_train.loc[train_fold_indices],
            y_train.loc[train_fold_indices],
            X_train.loc[eval_fold_indices],
            y_train.loc[eval_fold_indices]
        )

In [6]:
# TEST
if False:
    X_train_fold, y_train_fold, X_eval_fold, y_eval_fold = hf_read_real_data('medium', 81, 0, 0)
    print(X_train_fold['FLAG_OWN_CAR'].head(2))
    print(y_train_fold.head(2))
    print(X_eval_fold['FLAG_OWN_CAR'].head(2))
    print(y_eval_fold.head(2))    

### Read synthetic data (`hf_read_scenario_synthetic_data`)

Synthetic data for each scenario was created in Notebook `04_CTGAN_Data_Synthesis`. For very small samples, CTGAN can drop columns in synthetic data if they contain too few values. Therefore, to align the synthetic data with the real data, check for any columns missing from the synthetic data and replace them with a random value for categorical features, or 0 for numerical features. This will could create noise, but the column is be uninformative regardless and therefore does not warrant a more sophisticated approach.

In [7]:
import pandas as pd, numpy as np

def hf_read_scenario_synthetic_data(tier, ir, bootstrap, aug):
    '''Get the synthetic data for an augmentation level for a scenario.
    
    Summary
    -------
    Generate the path to the synthetic data from the input parameters then read the data from the
    parquet file into a DataFrame.
    '''

    # Get the path to the synthetic data
    scenario = f'{tier}-ir{ir:03d}-b{bootstrap:02}'
    synthetic_file = f'synthetic-aug{aug:03}.parquet'
    root = Path('..')
    synthetic_path = root / 'scenarios' / scenario / 'synthetic' / synthetic_file

    # Read the synthetic feature data direct into a DataFrame. Since only minority data is 
    # synthesised, create an array of positive labels the same length as the synthetic data.
    X_synthetic = pd.read_parquet(synthetic_path, engine='pyarrow')
    y_synthetic = pd.Series(np.ones(X_synthetic.shape[0], dtype=int), name='TARGET')

    # For extremely small samples, CTGAN may remove columns with a single value. Reinstate these
    # with a constant value so that the dataset remains aligned with the real data columns
    for col in X_train.columns:
        if col not in X_synthetic.columns:
            # Create the column with the correct dtype
            if X_train[col].dtype == 'category':
                # Fill with the first category or a placeholder - this will create noise
                first_cat = X_train[col].cat.categories[0]
                X_synthetic[col] = pd.Series([first_cat] * len(X_synthetic), dtype='category')
            else:
                # Numeric or other dtype: fill with median or 0 - this will create noise
                X_synthetic[col] = 0    
    
    # Align the categories for categorical columns with the training set categories
    for col in X_train.columns:
        if X_train[col].dtype == 'category':
            X_synthetic[col] = X_synthetic[col].cat.set_categories(
                X_train[col].cat.categories,
                ordered=X_train[col].cat.ordered)

    return X_synthetic, y_synthetic

In [8]:
# TEST
if False:
    X_syn_test, y_syn_test = hf_read_scenario_synthetic_data('small', 81, 0, 50)
    print(X_syn_test.head(2))
    print('\n', y_syn_test.head(2))
    print(X_syn_test['CODE_GENDER'])

In [9]:
# TEST
# Create a dummy snthetic data file for testing from the training set.
# Scenario: small-ir003-b00
if False:
    dummy_synthetic_data_50 = X_train.head(2793)
    dummy_synthetic_data_50.to_parquet(r"..\scenarios\small-ir003-b00\synthetic\synthetic-aug050.parquet", engine="pyarrow")

### Read scenario hyperparameters (`hf_read_scenario_hyperparameters`)

This cell reads the hyperparameters values which were determined during XGBoost hyperparameter tuning (Refer Notebook `07_XGB_Hyperparameter_Configuration`). Values are saved in separate files for each scenario. `scale_pos_weight` must be adjusted because it was tuned with a dataset with the nominal imbalance ratio for the scenario. Because `scale_pos_weight` is sensitive to IR and this changes for each fold after it is augmented with sample data, the value must be adjusted. The adjustment uses linear interpolation, which allow for values <1 if augmentation causes the positive class count to exceed the negative class count.

In [1]:
import json
from pathlib import Path

def hf_read_scenario_hyperparameters(tier, ir, n_neg, n_pos):
    '''
    Reads the XGBoost hyperparameters for the scenario from a JSON file.

    Parameters
    ----------
    tier : str : Name of the size tier e.g. 'large'.
    ir : int : Imbalance ratio.
    n_neg, n_pos : int : Number of negative and positive class samples.
    '''
    root = Path('..')
    hyperparameter_path = root / 'models' / 'xgboost' / 'hyperparameters'
    file = f'{tier}-ir{ir:03}.json'

    with open(hyperparameter_path / file, 'r') as f:
        params = json.load(f)

    # Need to adjust the `pos_weight` parameter because it was tuned to the nominal IR.
    # For balanced scenarios keep the original value to avoid division-by-zero error
    if ir != 1:
        params['scale_pos_weight'] = (params['scale_pos_weight']-1) / (ir-1) * (n_neg/n_pos-1) + 1

    # Set the maximum number of estimators based on total sample size
    size = n_neg + n_pos
    if size >= 80000:
        params['n_estimators'] = 1500
    elif size >= 40000:
        params['n_estimators'] = 1200
    else:
        params['n_estimators'] = 800
        
    return params

In [11]:
# TEST
hf_read_scenario_hyperparameters('medium', ir=81, n_neg=44139, n_pos=545)

{'colsample_bytree': 0.6,
 'learning_rate': 0.05,
 'reg_alpha': 0.5,
 'reg_lambda': 2.0,
 'subsample': 1.0,
 'scale_pos_weight': 40.49456422018349,
 'min_child_weight': 3,
 'max_depth': 4,
 'gamma': 0,
 'n_estimators': 1200}

### Preprocessor for numeric features (`preprocess`)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Create a preprocessor to replace missing with median for numeric columns only. 
# No need to call 'set_output(transform='panda')' because the preprocessor will not be working
# within GridSearchCV.
# NOTE: Float type columns are the only ones with missing values
numeric_cols = X_train.select_dtypes(include=['float64']).columns
preprocess = ColumnTransformer(
    transformers=[('median_imputer', SimpleImputer(strategy='median'), numeric_cols)],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

### Preprocessor for all features - Gaussian Naive Bayes (`preprocess_gnb`)

As with XGBoost, Gaussian naive Bayes requires missing numerical values to be imputed, but also requires values to be scaled and categorical values to be one-hotencoded.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identify column types
float_cols = X_train.select_dtypes(include=['float64']).columns
int_cols   = X_train.select_dtypes(include=['int64']).columns
bool_cols  = X_train.select_dtypes(include=['bool']).columns
cat_cols   = X_train.select_dtypes(include=['object', 'category']).columns

# Preprocessing pipeline
preprocess_gnb = ColumnTransformer(
    transformers=[
        # Floats: median impute + scale
        ('float_impute_scale',
         Pipeline([
             ('imputer', SimpleImputer(strategy='median')),
             ('scaler', StandardScaler())
         ]),
         float_cols),

        # Ints: scale (no missing values)
        ('int_scale',
         StandardScaler(),
         int_cols),

        # Booleans: to int
        ('bool_to_int', 
         FunctionTransformer(lambda x: x.astype(int)), 
         bool_cols),

        # Categoricals: one-hot encode (no missing values)
        ('cat_encode',
         OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         cat_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

### Compute best threshod for F1 score (`hf_compute_best_f1_threshold`)

In [14]:
from time import perf_counter
import numpy as np
from sklearn.metrics import (f1_score, precision_score, recall_score, average_precision_score,
    roc_auc_score, log_loss)

def hf_compute_best_f1_threshold(model, X_val, y_val, auc=False):
    '''
    Iterate over the range over the range of classification thresholds to find the one which gives
    the best overall F1 score. 

    Parameters
    ----------
    model : Trained XGBoost model
    X_val : pd.DataFrame of feature data
    y_val : pd.Series of labels
    auc : Boolean indicating whether AUC calculation is required

    Return
    ------
    tuple : dict of evaluation metric, array of predicted probabilities
    '''

    start = perf_counter()
    
    # Calculate the predicted probabilities from the feature data then iterate of the thresholds
    # to calculate the f1 scores. Select the best f1-score and corresonding threshold
    y_probs = model.predict_proba(X_val)[:, 1]
    thresholds = np.linspace(0, 1, 101)
    f1_scores = [f1_score(y_val, y_probs > t) for t in thresholds]
    best_threshold = float(thresholds[np.argmax(f1_scores)])
    best_f1 = max(f1_scores)

    # Calculate precision and recall
    y_pred = y_probs > best_threshold               # Need to recacculate using the best threshold
    best_pr = precision_score(y_val, y_pred)
    best_rec = recall_score(y_val, y_pred)
    threshold_tuning_runtime = (perf_counter() - start)/60
    
    # If required (e.g. for GaussianNB), calculate PR_AUC, ROC_AUC, and log loss
    if auc:    
        pr_auc = average_precision_score(y_val, y_probs)
        roc_auc = roc_auc_score(y_val, y_probs)

        # Check for absence of minority class
        if len(np.unique(y_val)) == 1:
            ll = np.nan
        else:
            ll = log_loss(y_val, y_probs)
        return (
            {
            'threshold_tuning_runtime': threshold_tuning_runtime,
            'best_threshold': best_threshold, 
            'best_f1': best_f1, 
            'best_rec': best_rec, 
            'best_pr': best_pr,
            'pr_auc': pr_auc,
            'roc_auc': roc_auc,
            'log_loss': ll},
            y_probs)        
    else:
        return (
            {
            'threshold_tuning_runtime': threshold_tuning_runtime,
            'best_threshold': best_threshold, 
            'best_f1': best_f1, 
            'best_rec': best_rec, 
            'best_pr': best_pr},
            y_probs)

### Read training results from the trained model (`hf_read_training_results`)

In [15]:
def hf_read_training_results(fold_model):
    '''Simple function to read specific attributes from a trained XGBoost model.'''

    # Read the results obtained for the validation fold from the trained model
    results = {}
    metric_results = fold_model.evals_result()['validation_0']
    results['n_estimators'] = fold_model.n_estimators
    results['early_stopping_rounds'] = fold_model.early_stopping_rounds

    # Read the best iteration then read the results for that iteration
    results['best_iteration'] = fold_model.best_iteration
    results['pr_auc'] = metric_results['aucpr'][results['best_iteration']]
    results['roc_auc'] = metric_results['auc'][results['best_iteration']]
    results['log_loss'] = metric_results['logloss'][results['best_iteration']]

    return results   

### Evaluate metrics from trained scenario model (`hf_evaluate_scenario_results`)

Because a common test set with fixed imbalance ratio (IR) is used for all scearnios, the sample weights must be adjusted to compensate for the difference in IR between the scenario and the test set. Evaluation metrics are calculated using both weighted and unweighted samples to validate this approach.

In [16]:
from sklearn.metrics import (f1_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, log_loss)

def hf_evaluate_scenario_results(scenario_model, threshold, X_test, y_test, ir):
    '''
    Calculate metrics from a trained scenario model using a defined threshold and test set. 
    Calculate both standard metrics and weighted metrics that adjust for the difference in the 
    class distributions of the scenario and the test set.    
    '''

    # Test set and scenairo proportions
    p_test_pos = (y_test == 1).mean()
    p_test_neg = 1 - p_test_pos
    p_scenario_pos = 1.0 / (1.0 + ir)
    p_scenario_neg = 1 - p_scenario_pos

    # Calculate the sample weights and assign to a weight vector
    w_pos = p_scenario_pos / p_test_pos
    w_neg = p_scenario_neg / p_test_neg
    w = np.where(y_test == 1, w_pos,  w_neg)
    
    # Calculate the test probabilities and associated predictions based on the threshold
    y_proba = scenario_model.predict_proba(X_test)[:, 1]
    y_preds = y_proba > threshold

    # Standard metrics
    results = {
        'pr_auc': average_precision_score(y_test, y_proba),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'log_loss': log_loss(y_test, y_proba),
        'threshold': threshold,
        'precision': precision_score(y_test, y_preds),
        'recall': recall_score(y_test, y_preds),
        'f1_score': f1_score(y_test, y_preds)
    }

    # Weighted metrics (not appropriate to weight pr_auc and roc_auc so ignore these in analysis)
    results.update({
        'pr_auc_w': average_precision_score(y_test, y_proba, sample_weight=w),   # Ignore
        'roc_auc_w': roc_auc_score(y_test, y_proba, sample_weight=w),            # Ignore
        'precision_w': precision_score(y_test, y_preds, sample_weight=w),
        'recall_w': recall_score(y_test, y_preds, sample_weight=w),
        'f1_score_w': f1_score(y_test, y_preds, sample_weight=w)
    })
    return results, y_proba

### Save the training and evaluation results to JSON (`hf_save_results`)

In [17]:
import json
from pathlib import Path
import numpy as np

def hf_save_results(tier, ir, bootstrap, fold, aug, results, y_true, y_proba, model_name=None):
    '''
    Write a dict of results from fold or scenario training to a JSON file. Save results from
    different models in different folders:
     - XGBoost fold-level:        fold-results
     - XGBoost scenario-level:    scenario-results
     - GaussianNB fold-level:     fold-results-gnb
     - GaussianNB scenario-level: scenario-results-gnb
    '''

    # Construct the target folder and filename
    root = Path('..') / 'scenarios' / f'{tier}-ir{ir:03}-b{bootstrap:02}'
    if (model_name is None) or (model_name == 'xgb'):
        path_suffix = ''
    else:
        path_suffix = f'-{model_name}'

    # Fold=-1 indicates scenario results
    if fold != -1:
        results_path = root / f'fold-results{path_suffix}'
        filename_results = f'results-fold{fold:1}-aug{aug:03}.json'
        filename_true = f'y-true-fold{fold:1}-aug{aug:03}.npy'
        filename_proba = f'y-proba-fold{fold:1}-aug{aug:03}.npy'
        filename_threshold = f'threshold-fold{fold:1}-aug{aug:03}.npy'
        threshold = results['best_threshold']
    else:
        results_path = root / f'scenario-results{path_suffix}'
        filename_results = f'results-aug{aug:03}.json'
        filename_true = f'y-true-aug{aug:03}.npy'
        filename_proba = f'y-proba-aug{aug:03}.npy'
        filename_threshold = f'threshold-aug{aug:03}.npy'
        threshold = results['threshold']

    # Create the target folder if it does not already exist save the results and probabilities
    results_path.mkdir(parents=True, exist_ok=True)
    
    # Save the results dict as a JSON file    
    with open(results_path / filename_results, 'w') as f:
        json.dump(results, f, indent=4)

    # Save y_true, y_probabilites, and the threshold as separate npy files
    np.save(results_path / filename_true, y_true)
    np.save(results_path / filename_proba, y_proba)
    np.save(results_path / filename_threshold, threshold)

In [18]:
# TEST
# Don't run after actual evaluation! This will overwrite the results.
#hf_save_results('small', 3, 0, 0, 50, {'a': 1})

### Find the best threshold and iterations from fold training (`hf_get_best_iteratons_and_threshold`)

For test set evaluation, select the median tuned threshold from the cross-validation (CV) folds. CV uses early stopping, while test set evaluation uses the median of best iterations from the CV folds.

In [19]:
import json, os
from pathlib import Path
import numpy as np

def hf_get_best_iterations_and_threshold(tier, ir, bootstrap, aug, files_expected, model_name=None):
    '''
    Read the best classifier iterations and threshold result for every fold for a given 
    scenario-augmentation level and calculate the medians.

    Parameters 
    ----------
    tier : str : Scenario size. E.g. `large`.
    ir : int : imbalance ratio.
    bootstrap : int : Bootstrap number.
    aug : int : Augmentation percentage.
    files_expected : The expected number of fold result files.
    '''

    # Search the scenario results folder for fold-level results and check that the expected
    # number of files were found
    root = Path('..')
    if model_name is None:
            results_path = root / 'scenarios' / f'{tier}-ir{ir:03}-b{bootstrap:02}' / 'fold-results'
    else:
            results_path = root / 'scenarios' / f'{tier}-ir{ir:03}-b{bootstrap:02}' / f'fold-results-{model_name}'
    all_files = os.listdir(results_path)

    # Get the file for the specific model. None for XGBoost or gnb for GaussianNB
    files = [f for f in all_files if f'{aug:03}.json' in f]
    
    files_found = len(files)
    #print('all_file:', all_files)
    #print('aug files:', files)
    assert files_found == files_expected, \
    f'Mismatch: expected {files_expected} results files, found {files_found}'

    # Read the best threshold and iteration for each fold from the JSON files then calcuate the
    # median.
    thresholds, best_iterations = [], []
    for file in files:
        with open(results_path / file, 'r') as f:
            results = json.load(f)

        # Best iterations are applicable to XGBoost but not GaussianNB
        if model_name is None:
            best_iterations.append(results['best_iteration'])
        thresholds.append(results['best_threshold'])

    if model_name is None:
        return int(np.median(best_iterations)), np.median(thresholds).item()
    else:
        return None, np.median(thresholds).item()

In [20]:
# TEST
#threshold, iteration = hf_get_best_iterations_and_threshold('small', ir=81, bootstrap=0, aug=100, files_expected=5)
#print(iteration, threshold)

## Main functions for fold and scenario training and evaluation

### Train and evaluate an augmented fold (`train_and_evaluate_fold`)

Each fold for each scenario is processed individually and results are saved then collated in a separate notebook. The training fold is processed to impute missing values then combined with synthetic data. The validation fold is processed using the imputer fit on the training fold. By contrast, CTGAN is trained on scenario data which has been imputed using a preprocessor fit on the full scenario dataset, therefore it does not require further preprocessing in this function.  

This cell trains and evaluates an XGBoost classifier using a single training fold of data, with results saved and collated in function. An imputer is fit on the training fold the applied to the training and validation folds. The synthetic data was generated from a CTGAN model that was trained on the full dataset for the scenario, and that had missing values already imputed and therefore does not need imputing. The indices for the fold, the synthetic data, and the XGBoost hyperparameters are read using helper functions. The datasets for training are stored as DataFrames so that categorical variables can be enabled in XGBoost.

While hyperparameters are read from JSON files, `scale_pos_weight` must be adjusted based on the imbalance ratio of the fold data after augmentation with synthetic data. Therefore, the negative and positive class counts are passed to the helper function. Given sufficient augmentation, the IR can fall below 1 causing the `scale_pos_weight` value also to fall below 1.

To help improve stability across scenarios, CV training uses early stopping. The maximum number of iterations is set based on the sample size. PR-AUC, ROC-AUC, and log-loss are read from the classifier after training.

In [21]:
def hf_check_fold_dependencies():
    # Check that required datasets are in memory and functions exist
    assert X_train is not None, f'Traning feature dataset not loaded'
    assert y_train is not None, f'Training label dataset not loaded'
    assert hf_read_real_data
    assert preprocess
    assert hf_read_scenario_synthetic_data
    assert hf_read_training_results
    assert hf_read_scenario_hyperparameters
    assert hf_compute_best_f1_threshold
    assert hf_save_results

In [22]:
from time import perf_counter
import pandas as pd
from xgboost import XGBClassifier

# Global variable for testing
test_models = {}

def train_and_evaluate_fold(tier, ir, bootstrap, fold, aug, isreal=False):
    '''
    Train a fold-level dataset using the specified level of data augmentation.
    Parameters
    ----------
    tier : str
        Size tier e.g. `large`.
    ir : int
        Imbalance ratio e.g. 3 indicating 1:3.
    bootstrap : int
        Bootstrap number e.g. 0.
    fold : int
        Cross validation fold number.
    aug : int
        Augmentation level percentage e.g. 50 indicating 50%.
    isreal : boolean
        Whether the augmentation dataset is real data. Only true for the real data experiment.

    Summary
    -------
    1. Load the scenario data 
    2. Impute missing value in train then test
    3. Load the synthetic data
    4. Concatenate the real and synthetic data
    5. Train XGBBoost
    6. Evaluate the fold and find the best threshold
'''
    # Check required data and functions have been loaded
    hf_check_fold_dependencies()
    augmented = True if aug > 0 else False
    
    # Read the real fold data
    X_train_fold, y_train_fold, X_val_fold, y_val_fold = hf_read_real_data(
        tier, ir, bootstrap, fold)
    n_neg_real = (y_train_fold == 0).sum()     # Majority class
    n_pos_real = (y_train_fold == 1).sum()     # Minority class

    # Impute the training and test sets. Do not imput the synthetic data because the GAN was
    # trained using imputed values for the whole scenario.
    X_train_fold_imp = preprocess.fit_transform(X_train_fold)
    X_val_fold_imp = preprocess.transform(X_val_fold)

    # Augment the real data with the synthetic data. Convert the synthetic label data to a numpy
    # array to match the training label data. Skip if augmentation level = 0
    if augmented:
        X_synthetic, y_synthetic = hf_read_scenario_synthetic_data(tier, ir, bootstrap, aug)

        # Check whether the synthetic dataset contains any records. For small datasets and 
        # augmentation levels, the resulting number of records could be <1.
        if len(X_synthetic) != 0:
            X_train_fold_imp_aug = pd.concat([X_train_fold_imp, X_synthetic])
            y_train_fold_aug = np.concatenate([y_train_fold, y_synthetic.to_numpy()])
        else:
            augmented = False

    # If augmentation if not required, simply redirect the augmented variables
    if not augmented:
        X_train_fold_imp_aug = X_train_fold_imp
        y_train_fold_aug = y_train_fold
    
    # Train the classifier using tuned hyperparameters read from a JSON file.
    n_neg = (y_train_fold_aug == 0).sum()     # Majority class (except for high minority augmentation)
    n_pos = (y_train_fold_aug == 1).sum()     # Minority class

    fixed_params = {
        'tree_method': 'hist',
        'objective': 'binary:logistic',
        'enable_categorical': True,
        'eval_metric': ['auc', 'logloss', 'aucpr'],  # List aucpr last to determine early stopping
        'nthread': 4,        
        'early_stopping_rounds': 100,
        'verbosity': 0}
    tuned_params = hf_read_scenario_hyperparameters(tier, ir, n_neg, n_pos)
    xgb_params = fixed_params | tuned_params
    fold_model = XGBClassifier(**xgb_params)
   
    fit_start = perf_counter()                     # Timer start
    fold_model.fit(
        X_train_fold_imp_aug, y_train_fold_aug, 
        eval_set=[(X_val_fold_imp, y_val_fold)],
        verbose=False)
    fit_runtime = (perf_counter() - fit_start)/60  # Time end

    # Read the required evaluation metrics from the classifier
    results = hf_read_training_results(fold_model)

    # Tune the classifier threshold and calculate the best F1, precision, and recall
    threshold_results, y_val_fold_proba = hf_compute_best_f1_threshold(
        fold_model, 
        X_val_fold_imp, 
        y_val_fold)
    results.update(threshold_results)

    # Save the fold metadata
    results['n_train'] = y_train_fold.shape[0]
    results['nominal_ir'] = ir
    results['augmentation_pct'] = aug
    results['fold_number'] = fold
    results['n_neg_real'] = n_neg_real.item()
    results['n_pos_real'] = n_pos_real.item()
    results['n_synthetic'] = y_synthetic.shape[0] if augmented else 0
    results['n_train_aug'] = y_train_fold_aug.shape[0]
    results['n_train_aug_neg'] = n_neg.item()
    results['n_train_aug_pos'] = n_pos.item()
    results['augmented_ir'] = (n_neg/n_pos).item() if augmented else ir
    results['fit_runtime'] = fit_runtime

    # Save and return the results including the y values and predicted probabilities
    hf_save_results(tier, ir, bootstrap, fold, aug, results, y_val_fold, y_val_fold_proba)
    return results

In [23]:
# TEST
#_ = train_and_evaluate_fold('medium', 81, 0, 1, 50)
#train_and_evaluate_fold('small', 81, 0, 0, 100)

In [24]:
# TEST
# Using experimentatal data
# for fold in range(2):
#    train_and_evaluate_fold('medium', 81, 0, fold, 0)

### Train and evaluate an augmented scenario (`train_and_evaluate_scenario`)

Scenario-level training uses the full scenario training dataset and evaluates the classifier against the test set. Because the test set is used to evaluate the model instead of using cross validation, the model is trained without using an evaluation set - preventing early stopping to be used. Instead, the model sets `n_estimators` to the median of the best iterations found for the folds. Similarly, the best threshold is found from fold training and used to evaluate the model against the test set.

In [25]:
def hf_check_dependencies():
    # Check that required datasets are in memory and functions exist
    assert X_train is not None, f'Traning feature dataset not loaded'
    assert y_train is not None, f'Training label dataset not loaded'
    assert X_test is not None,  f'Test feature dataset not loaded'
    assert y_test is not None,  f'Test labal dataset not loaded'
    assert hf_read_real_data, 'Function hf_read_fold_real_data not loaded'
    assert preprocess
    assert hf_read_scenario_synthetic_data
    assert hf_read_scenario_hyperparameters
    assert hf_evaluate_scenario_results
    assert hf_get_best_iterations_and_threshold
    assert hf_save_results

In [26]:
from time import perf_counter
import pandas as pd
from xgboost import XGBClassifier

def train_and_evaluate_scenario(tier: str, ir: int, bootstrap: int, aug: int) -> dict:
    '''
    Train a fold-level dataset using the specified level of data augmentation.
    Parameters
    ----------
    tier : str
        Size tier e.g. `large`.
    ir : int
        Imbalance ratio e.g. 3 indicating 1:3.
    bootstrap : int
        Bootstrap number e.g. 0.
    aug : int
        Augmentation level percentage e.g. 50 indicating 50%.
'''
    hf_check_dependencies()
    files_expected = 5       # Number of fold-level results files that should be available
    augmented = True if aug > 0 else False

    # Read the real fold data
    X_train_scenario, y_train_scenario = hf_read_real_data(tier, ir, bootstrap, -1)
    n_neg_real = (y_train_scenario == 0).sum()     # Majority class
    n_pos_real = (y_train_scenario == 1).sum()     # Minority class

    # Impute the training and test sets. Do not imput the synthetic data because the GAN was
    # trained using imputed values for the whole scenario.
    X_train_scenario_imp = preprocess.fit_transform(X_train_scenario)
    X_test_imp = preprocess.transform(X_test)

    # Augment the real data with the synthetic data. Convert the label training data to a numpy
    # array to match the synthetic label data. Skip if augmentation level = 0
    if augmented:
        X_synthetic, y_synthetic = hf_read_scenario_synthetic_data(tier, ir, bootstrap, aug)

        if len(X_synthetic) !=0:
            X_train_scenario_imp_aug = pd.concat([X_train_scenario_imp, X_synthetic])
            y_train_scenario_aug = np.concatenate([y_train_scenario, y_synthetic.to_numpy()])
        else:
            augmented = False

    if not augmented:
        X_train_scenario_imp_aug = X_train_scenario_imp
        y_train_scenario_aug = y_train_scenario

   # Get the median best threshold obtained from training the folds. Use the 
    n_estimators, threshold = hf_get_best_iterations_and_threshold(
        tier, ir, bootstrap, aug, files_expected)
    #print('threshold:', thresholds, 'n_estimators:', n_estimators)
        
    # Train the classifier using tuned hyperparameters read from a JSON file.
    # Instead of early stopping, disable and specify n_estimators as the median of best iterations
    # for fold training.
    n_neg = (y_train_scenario_aug == 0).sum()     # Majority class - except for high minority augmentation
    n_pos = (y_train_scenario_aug == 1).sum()     # Minority class
    fixed_params = {
        'tree_method': 'hist',
        'objective': 'binary:logistic',
        'enable_categorical': True,
        'eval_metric': ['aucpr', 'auc', 'logloss'],
        'nthread': 4,
        #'early_stopping_rounds': 100,            # Disable for scenario training
        'verbosity': 0}
    tuned_params = hf_read_scenario_hyperparameters(tier, ir, n_neg, n_pos)
    tuned_params['n_estimators'] = n_estimators   # Override the ceiling value used for folds
    xgb_params = fixed_params | tuned_params
    scenario_model = XGBClassifier(**xgb_params)
   
    fit_start = perf_counter()                     # Timer start
    scenario_model.fit(
        X_train_scenario_imp_aug, y_train_scenario_aug, 
        # Do not use eval_set
        verbose=False)
    fit_runtime = (perf_counter() - fit_start)/60  # Timer end

    # Evaluate the model against the test set using the best threshold from fold training
    # Pass the IR of the training set to the evaluation function and not the augmented training set
    results, y_test_proba = hf_evaluate_scenario_results(
        scenario_model, 
        threshold, 
        X_test_imp, 
        y_test, ir)

    # Save the scenario metadata
    results['n_estimators'] = n_estimators
    results['n_train'] = y_train_scenario.shape[0]
    results['nominal_ir'] = ir
    results['augmentation_pct'] = aug
    results['n_neg_real'] = n_neg_real.item()
    results['n_pos_real'] = n_pos_real.item()
    results['n_synthetic'] = y_synthetic.shape[0] if augmented else 0
    results['n_train_aug'] = y_train_scenario_aug.shape[0]
    results['n_train_aug_neg'] = n_neg.item()
    results['n_train_aug_pos'] = n_pos.item()
    results['augmented_ir'] = (n_neg/n_pos).item() if augmented else ir
    results['fit_runtime'] = fit_runtime

    # Log the results - Pass fold=-1 to indicate a scenario instead of fold
    hf_save_results(tier, ir, bootstrap, -1, aug, results, y_test, y_test_proba)

    return results

In [27]:
# TEST
#from pprint import pprint
#results = train_and_evaluate_scenario('medium', 81, 0, 50)
#train_and_evaluate_scenario('small', 3, 0, 0)
#pprint(results)

### Wrapper function for fold training (`hf_fold_training`)

In [28]:
import os
from time import perf_counter

def hf_fold_training(tier, ir, augs, n_folds=5, testmode=False, isreal=False):
    '''
    Wrapper function to call `train_and_evaluate_fold`.
    '''
    # Call the fold training function and display progress by fold
    start = perf_counter()
    print(f'Train and evaluate fold-level ({tier:6}, {ir:03d}):\n  ', end='')
    
    for aug in augs:
        # Check for the stop-signal
        if os.path.exists('..\\STOP.txt'):
            print("Stop signal detected. Model training halted.")
        else:
            # Do not apply augmentation to balanced scenarios
            if not((ir==1) and (aug>0)):
                print(f'Aug{aug:d}', end='')
                for fold in range(n_folds):
                    print('.', end='')
                    if not testmode:
                        train_and_evaluate_fold(tier, ir, 0, fold, aug, isreal)
                        pass            
                else:
                    print(' ', end='')
    else:
        print(f'  | Scenario runtime: {(perf_counter() - start) / 60:5.2f} minutes')
        if testmode:
            print(' ** TEST MODE MESSAGE: Training was not completed **')

### Wrapper function for scenario training (`hf_scenario_training`)

In [29]:
import os
from time import perf_counter

def hf_scenario_training(tier, ir, augs, testmode=False):
    '''
    Train and scenarios with and without augmentation. Scenario are evaluated against a
    holdout test set instead of using cross validation.
    '''
    start = perf_counter()
    print(f'Train and evaluate scenario-level ({tier:6}, {ir:03d}):\n  ', end='')
    for aug in augs:
        # Check for the stop-signal
        if os.path.exists('..\\STOP.txt'):
            print("Stop signal detected. Model training halted.")
        else:
            # Skip augmented training if scenario is balanced
            if not((ir == 1) and (aug != 0)):
                print(f'Aug{aug:d}', end='')
                if not testmode:
                    pass
                    train_and_evaluate_scenario(tier, ir, 0, aug)
                print('. ', end='')
    else:
        print(f'  | Scenario runtime: {(perf_counter() - start)/60:5.1f} minutes')
        if testmode:
            print('TESTMODE MESSAGE: Training was not completed.')

### Combined fold and scenario training (`hf_fold_scenario_training`)

In [31]:
from time import perf_counter

def hf_fold_scenario_training(scenarios):
    '''
    Call the XGBoost fold and scenario training wrapper functions for scenarios.

    Parameters
    ----------
    scenarios : dict like {'tier': {'ir': [], 'augs': []}}
    '''
    start = perf_counter()
    
    # Iterate through the tiers, IRs, and augs, and call fold-level and sceanrio-level training
    # functions
    for tier, params in scenarios.items():
        augs = params['augs']
    
        for ir in params['irs']:
            print(f'Training {tier} tier IR {ir} with augs {augs}...')
            pass
            hf_fold_training(tier, ir, augs, testmode=False)
            hf_scenario_training(tier, ir, augs, testmode=False)
    
    print(f'\nTotal fold- and scenario-level runtime: {(perf_counter() - start)/60:5.1f} minutes')
    print('\n-- All scenarios complete --')

## Prepare scenarios before model training

Because several experiments were added to the original grid and because CTGAN models for scenarios were trained in stages, it is helpful to check that all required models and synthetic data sets exist before training classifiers. Following this, scenarios are split into five groups which are executed separately to avoid a very long-running process.

### Check all required data is available for model training

Iterate through the scenarios and confirm which CTGAN models which synthetic datasets have been created.
```
..
    |-- models
        |-- ctgan
            |-- synthesizer-large-ir-003-b00.pkl
            |-- ...
    |-- scenarios
        |-- <tier>-ir<IR>-b<bootstrap>
            |-- synthetic
                |-- synthetic-aug<augmentation percentage>.parquet
                |-- ...
        |-- ...
```

In [32]:
from pathlib import Path
from itertools import product

# Check the all CTGAN models have been trained for the imbalanced scenarios

# CTGAN models
# Iterate through files and list them by tier
print('\nCTGAN SYNTHESIZERS')
path = Path(f'..\\models\\ctgan\\')
files = list(path.glob(f'synthesizer*.pkl'))
tier = ''
for f in files:
    parts = f.stem.split('-')
    if parts[1] != tier:
        print()
        tier = parts[1]
    print('-'.join(parts[1:3]), '', end='')
print()

# Synthetic data
# Iterate through the scenarios and list the augmentation levels for synthetic data
print('\nSYNTHETIC DATA\n')
path = Path(f'..\\scenarios\\')
files = path.glob('*')
for dir in path.iterdir():
    print(dir.stem, ':', sep='', end='')
    files = [f.stem.split('-')[1] for f in (dir / 'synthetic').glob('synthetic*.parquet')]
    print(files)
print()


CTGAN SYNTHESIZERS

large-ir003 large-ir009 large-ir027 large-ir081 large-ir243 
medium-ir001 medium-ir003 medium-ir009 medium-ir027 medium-ir081 medium-ir243 medium-ir729 
small-ir003 small-ir009 small-ir027 small-ir081 small-ir243 
smalla-ir001 smalla-ir003 smalla-ir009 
smallb-ir001 smallb-ir003 smallb-ir009 
smallc-ir001 smallc-ir003 smallc-ir009 
smalld-ir001 smalld-ir003 smalld-ir009 
xsmalla-ir001 xsmalla-ir003 xsmalla-ir009 xsmalla-ir027 xsmalla-ir081 
xsmallb-ir001 xsmallb-ir003 xsmallb-ir009 xsmallb-ir027 xsmallb-ir081 
xsmallc-ir001 xsmallc-ir003 xsmallc-ir009 xsmallc-ir027 xsmallc-ir081 
xsmalld-ir001 xsmalld-ir003 xsmalld-ir009 xsmalld-ir027 

SYNTHETIC DATA

large-ir001-b00:[]
large-ir003-b00:['aug050', 'aug100', 'aug200', 'aug400']
large-ir009-b00:['aug050', 'aug100', 'aug200', 'aug400']
large-ir027-b00:['aug050', 'aug100', 'aug200', 'aug400']
large-ir081-b00:['aug050', 'aug100', 'aug200', 'aug400']
large-ir243-b00:['aug050', 'aug100', 'aug200', 'aug400']
medium-ir001-b

### Define dicts of scenarios

In [33]:
# 1. Main experiments
scenarios_main = {
    'large': {'irs': [3, 9, 27, 81, 243], 'augs': [0, 50, 100, 200, 400]},
    'medium': {'irs': [1, 3, 9, 27, 81, 243, 729], 'augs': [0, 50, 100, 200, 400]},
    'small': {'irs': [1, 3, 9, 27, 81, 243], 'augs': [0, 50, 100, 200, 400]}
     }

# 2. Increase the grain of the augmentation level
scenarios_fine_grained = {
    'medium': {'irs': [3, 9, 27, 81], 'augs': [4, 8, 13, 18, 22, 28, 33, 38, 44]}
}

# 3. Double the maximum augmentation percentage
scenarios_augmentation_800 = {
    'medium': {'irs': [243, 729], 'augs': [800]},
    'small': {'irs': [243], 'augs': [800]}
}

# 4. Augmentation with real data instead of synthetic data
scenarios_real = {'smallaug': {'irs': [1, 3, 9, 27, 81], 
                               'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]}
}

# 5. Very small and extra small scenarios
scenarios_extra_small = {
    'smalla': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'smallb': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'smallc': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'smalld': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'xsmalla': {'irs': [1, 3, 9, 27], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'xsmallb': {'irs': [1, 3, 9, 27], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'xsmallc': {'irs': [1, 3, 9, 27], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'xsmalld': {'irs': [1, 3, 9, 27], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]}
}

## XGBoost training for all scenarios

All scenario lists to be trained:
```
hf_fold_scenario_training_gnb(scenarios_main)
hf_fold_scenario_training_gnb(scenarios_fine_grained)
hf_fold_scenario_training_gnb(scenarios_augmentation_800)
hf_fold_scenario_training_gnb(scenarios_real)
hf_fold_scenario_training_gnb(scenarios_extra_small)

### 1. Main scenarios IR 1-243, augmentation 0-400%

In [34]:
hf_fold_scenario_training(scenarios_main)

Training large tier IR 3 with augs [0, 50, 100, 200, 400]...
Train and evaluate fold-level (large , 003):
  Aug0..... Aug50..... Aug100..... Aug200..... Aug400..... Aug400.....   | Scenario runtime:  2.73 minutes
Train and evaluate scenario-level (medium, 243):
  Aug0. Aug50. Aug100. Aug200. Aug400.   | Scenario runtime:   0.2 minutes
Training medium tier IR 729 with augs [0, 50, 100, 200, 400]...
Train and evaluate fold-level (medium, 729):
  Aug0..... Aug50..... Aug100..... Aug200..... Aug400.....   | Scenario runtime:  2.80 minutes
Train and evaluate scenario-level (medium, 729):
  Aug0. Aug50. Aug100. Aug200. Aug400.   | Scenario runtime:   0.2 minutes
Training small tier IR 1 with augs [0, 50, 100, 200, 400]...
Train and evaluate fold-level (small , 001):
  Aug0.....   | Scenario runtime:  0.57 minutes
Train and evaluate scenario-level (small , 001):
  Aug0.   | Scenario runtime:   0.1 minutes
Training small tier IR 3 with augs [0, 50, 100, 200, 400]...
Train and evaluate fold-lev

### 2. Finegrained augmentation interval 0-50%

In [109]:
hf_fold_scenario_training(scenarios_fine_grained)

Training medium tier IR 3 with augs [4, 8, 13, 18, 22, 28, 33, 38, 44]...
Train and evaluate fold-level (medium, 003):
  Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44.....   | Scenario runtime:  5.91 minutes
Train and evaluate scenario-level (medium, 003):
  Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44.   | Scenario runtime:   0.9 minutes
Training medium tier IR 9 with augs [4, 8, 13, 18, 22, 28, 33, 38, 44]...
Train and evaluate fold-level (medium, 009):
  Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44.....   | Scenario runtime:  4.57 minutes
Train and evaluate scenario-level (medium, 009):
  Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44.   | Scenario runtime:   0.6 minutes
Training medium tier IR 27 with augs [4, 8, 13, 18, 22, 28, 33, 38, 44]...
Train and evaluate fold-level (medium, 027):
  Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33.....

### 3. 800% Augmentation

In [108]:
hf_fold_scenario_training(scenarios_augmentation_800)

Training medium tier IR 243 with augs [800]...
Train and evaluate fold-level (medium, 243):
  Aug800.....   | Scenario runtime:  0.39 minutes
Train and evaluate scenario-level (medium, 243):
  Aug800.   | Scenario runtime:   0.0 minutes
Training medium tier IR 729 with augs [800]...
Train and evaluate fold-level (medium, 729):
  Aug800.....   | Scenario runtime:  0.39 minutes
Train and evaluate scenario-level (medium, 729):
  Aug800.   | Scenario runtime:   0.0 minutes
Training small tier IR 243 with augs [800]...
Train and evaluate fold-level (small , 243):
  Aug800.....   | Scenario runtime:  0.22 minutes
Train and evaluate scenario-level (small , 243):
  Aug800.   | Scenario runtime:   0.0 minutes

Total fold- and scenario-level runtime:   1.1 minutes

-- All scenarios complete --


### 4. Real data augmentation

In [ ]:
hf_fold_scenario_training(scenarios_real)

Training smallaug tier IR 1 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]...
Train and evaluate fold-level (smallaug, 001):
  Aug0.....   | Scenario runtime:  0.57 minutes
Train and evaluate scenario-level (smallaug, 001):
  Aug0.   | Scenario runtime:   0.1 minutes
Training smallaug tier IR 3 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]...
Train and evaluate fold-level (smallaug, 003):
  Aug0..... Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44..... Aug50..... Aug100..... Aug200.....   | Scenario runtime:  9.44 minutes
Train and evaluate scenario-level (smallaug, 003):
  Aug0. Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44. Aug50. Aug100. Aug200.   | Scenario runtime:   1.3 minutes
Training smallaug tier IR 9 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]...
Train and evaluate fold-level (smallaug, 009):
  Aug0..... Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Au

### 5. Extra-small scenarios

In [36]:
hf_fold_scenario_training(scenarios_extra_small)

. Aug50.....   | Scenario runtime:  1.94 minutes
Train and evaluate scenario-level (smallb, 009):
  Aug0. Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44. Aug50.   | Scenario runtime:   0.2 minutes
Training smallc tier IR 1 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]...
Train and evaluate fold-level (smallc, 001):
  Aug0.....   | Scenario runtime:  0.19 minutes
Train and evaluate scenario-level (smallc, 001):
  Aug0.   | Scenario runtime:   0.0 minutes
Training smallc tier IR 3 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]...
Train and evaluate fold-level (smallc, 003):
  Aug0..... Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44..... Aug50.....   | Scenario runtime:  1.90 minutes
Train and evaluate scenario-level (smallc, 003):
  Aug0. Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44. Aug50.   | Scenario runtime:   0.2 minutes
Training smallc tier IR 9 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]...

C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.con

.

C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.con

.

...

. Aug8

C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.con

. Aug13

C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_synthetic[col] = 0
C:\Users\daveh\AppData\Local\Temp\ipykernel_19156\3427870481.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.con

. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44. Aug50.   | Scenario runtime:   0.1 minutes

Total fold- and scenario-level runtime:  41.4 minutes

-- All scenarios complete --


### Test - View a sample of fold and scenarios results

In [111]:
# TEST
import numpy as np

if True:
    # Fold results
    filename_true = '..\\scenarios\\small-ir001-b00\\fold-results\\y-true-fold0-aug000.npy'
    filename_proba = '..\\scenarios\\small-ir001-b00\\fold-results\\y-proba-fold0-aug000.npy'
    filename_threshold = '..\\scenarios\\small-ir001-b00\\fold-results\\threshold-fold0-aug000.npy'
    y_true = np.load(filename_true)
    y_proba = np.load(filename_proba)
    threshold = float(np.load(filename_threshold))
    
    print('true\n:', y_true[:50])
    print('y_proba:\n', y_proba[:10])
    print('threshold:', threshold)

    # Scenario results
    filename_true = '..\\scenarios\\small-ir001-b00\\scenario-results\\y-true-aug000.npy'
    filename_proba = '..\\scenarios\\small-ir001-b00\\scenario-results\\y-proba-aug000.npy'
    filename_threshold = '..\\scenarios\\small-ir001-b00\\scenario-results\\threshold-aug000.npy'
    y_true = np.load(filename_true)
    y_proba = np.load(filename_proba)
    threshold = float(np.load(filename_threshold))
    
    print('true\n:', y_true[:50])
    print('y_proba:\n', y_proba[:10])
    print('threshold:', threshold)

true
: [0 0 0 0 1 0 0 0 0 1 0 1 1 1 1 1 1 0 1 0 0 1 0 0 0 1 0 0 1 0 1 0 0 1 0 1 0
 0 0 0 0 0 0 1 0 0 1 1 0 1]
y_proba:
 [0.09952302 0.6672138  0.19261917 0.69629055 0.6520603  0.13524656
 0.296847   0.44608182 0.23531882 0.8918908 ]
threshold: 0.2
true
: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0]
y_proba:
 [0.10078559 0.6413773  0.38361728 0.4886793  0.07022864 0.13268381
 0.37145025 0.310272   0.24598195 0.06076827]
threshold: 0.24


## Scenario training and evaluation using `GaussianNB`

This section implements GaussianNB model training, and follows the same methodology as XGBoost in the previous section.

### Fold level GaussianNB (`train_and_evaluate_gnb_fold`)

In [30]:
from time import perf_counter
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score, precision_recall_curve
import numpy as np

# Global variable for testing
test_models = {}

def train_and_evaluate_gnb_fold(tier: str, ir: int, bootstrap: int, fold: int, aug: int) -> dict:
    '''
    Train a fold-level dataset using GaussianNB and with the specified level of data augmentation.
    Parameters
    ----------
    tier : str
        Size tier e.g. `large`.
    ir : int
        Imbalance ratio e.g. 3 indicating 1:3.
    bootstrap : int
        Bootstrap number e.g. 0.
    fold : int
        Cross validation fold number.
    aug : int
        Augmentation level percentage e.g. 50 indicating 50%.

    1. Load the scenario data
    2. Impute missing value in train then test
    3. Load the synthetic data
    4. Augment with the synthetic data
    5. Train XGBBoost
    6. Evaluate against the test set using the best threshold
'''
    
    # Check required data and functions have been loaded
    hf_check_fold_dependencies()
    augmented = True if aug > 0 else False
    
    # Read the real fold data
    X_train_fold, y_train_fold, X_val_fold, y_val_fold = hf_read_real_data(
        tier, ir, bootstrap, fold)
    n_neg_real = (y_train_fold == 0).sum()     # Majority class
    n_pos_real = (y_train_fold == 1).sum()     # Minority class

    # Impute the training and test sets. Do not imput the synthetic data because the GAN was
    # trained using imputed values for the whole scenario.
    X_train_fold_imp = preprocess_gnb.fit_transform(X_train_fold).copy()
    X_val_fold_imp = preprocess_gnb.transform(X_val_fold).copy()

    # Augment the real data with the synthetic data. Convert the synthetic label data to a numpy
    # array to match the training label data. Skip if augmentation level = 0
    if augmented:
        X_synthetic, y_synthetic = hf_read_scenario_synthetic_data(tier, ir, bootstrap, aug)

        # Check whether the synthetic dataset contains any records. For small datasets and 
        # augmentation levels, the resulting number of records could be <1.
        if len(X_synthetic) != 0:

            # For GaussianNB preprocssing of the synthetic data is required for dtype conversion and
            # OHE. Imupting will have no effect because CTGAN did not produce any missing values.
            X_synthetic_imp = preprocess_gnb.transform(X_synthetic).copy()
            X_train_fold_imp_aug = pd.concat([X_train_fold_imp, X_synthetic_imp])
            y_train_fold_aug = np.concatenate([y_train_fold, y_synthetic.to_numpy()])
        else:
            augmented = False
        
    if not augmented:
        X_train_fold_imp_aug = X_train_fold_imp
        y_train_fold_aug = y_train_fold

    # Count the augmented class samples
    n_neg = (y_train_fold_aug == 0).sum()     # Majority class (except for high minority augmentation)
    n_pos = (y_train_fold_aug == 1).sum()     # Minority class

    # Train GaussianNB model
    start = perf_counter()
    fold_model = GaussianNB()
    fold_model.fit(X_train_fold_imp_aug, y_train_fold_aug)
    fit_runtime = (perf_counter() - start) / 60

    # Tune the classifier threshold and calculate the best F1, precision, and recall
    results, y_val_fold_proba = hf_compute_best_f1_threshold(
        fold_model, 
        X_val_fold_imp, 
        y_val_fold, 
        auc=True)

    # Assume GaussianNB is only run after XGBoost so metadata is not required. But store it 
    # initially anyway.
    # Save the fold metadata
    results['n_train'] = y_train_fold.shape[0]
    results['nominal_ir'] = ir
    results['augmentation_pct'] = aug
    results['fold_number'] = fold
    results['n_neg_real'] = n_neg_real.item()
    results['n_pos_real'] = n_pos_real.item()
    results['n_synthetic'] = y_synthetic.shape[0] if augmented else 0
    results['n_train_aug'] = y_train_fold_aug.shape[0]
    results['n_train_aug_neg'] = n_neg.item()
    results['n_train_aug_pos'] = n_pos.item()
    results['augmented_ir'] = (n_neg/n_pos).item() if augmented else ir
    results['fit_runtime'] = fit_runtime

    # Save and return the results
    hf_save_results(tier, ir, bootstrap, fold, aug, 
                    results, y_val_fold, y_val_fold_proba, 
                    model_name='gnb')   
    return results

### Scenario level GaussianNB (`train_and_evaluate_gnb_scenario`)

In [31]:
from time import perf_counter
import pandas as pd

def train_and_evaluate_gnb_scenario(tier: str, ir: int, bootstrap: int, aug: int) -> dict:
    '''
    Train a fold-level dataset using the specified level of data augmentation.
    Parameters
    ----------
    tier : str
        Size tier e.g. `large`.
    ir : int
        Imbalance ratio e.g. 3 indicating 1:3.
    bootstrap : int
        Bootstrap number e.g. 0.
    aug : int
        Augmentation level percentage e.g. 50 indicating 50%.
'''
    model_name = 'gnb'
    hf_check_dependencies()
    files_expected = 5       # Number of fold-level results files that should be available
    augmented = True if aug > 0 else False

    # Read the real fold data
    X_train_scenario, y_train_scenario = hf_read_real_data(tier, ir, bootstrap, -1)
    n_neg_real = (y_train_scenario == 0).sum()     # Majority class
    n_pos_real = (y_train_scenario == 1).sum()     # Minority class

    # Impute the training and test sets. Do not imput the synthetic data because the GAN was
    # trained using imputed values for the whole scenario.
    X_train_scenario_imp = preprocess_gnb.fit_transform(X_train_scenario)
    X_test_imp = preprocess_gnb.transform(X_test)

    # Augment the real data with the synthetic data. Convert the label training data to a numpy
    # array to match the synthetic label data. Skip if augmentation level = 0
    if augmented:
        X_synthetic, y_synthetic = hf_read_scenario_synthetic_data(tier, ir, bootstrap, aug)

        # For GaussianNB preprocssing of the synthetic data is required for dtype conversion and
        # OHE. Imupting will have no effect because CTGAN did not produce any missing values.
        X_synthetic_imp = preprocess_gnb.transform(X_synthetic)

        if len(X_synthetic) !=0:
            X_train_scenario_imp_aug = pd.concat([X_train_scenario_imp, X_synthetic_imp])
            y_train_scenario_aug = np.concatenate([y_train_scenario, y_synthetic.to_numpy()])
            # TEST
            #print('X_train_scenario:', X_train_scenario.shape, 'y_train_scenario:', y_train_scenario.shape)
            #print('X_synthetic:', X_synthetic.shape, 'y_synthetic:', y_synthetic.shape)
            #print('X_aug      :', X_train_scenario_imp_aug.shape, 'y_train_scenario_aug:', y_train_scenario_aug.shape)

        else:
            augmented = False
    if not augmented:
        X_train_scenario_imp_aug = X_train_scenario_imp
        y_train_scenario_aug = y_train_scenario

    # Count the augmented class samples
    n_neg = (y_train_scenario_aug == 0).sum()     # Majority class (except for high minority augmentation)
    n_pos = (y_train_scenario_aug == 1).sum()     # Minority class

   # Get the median best threshold obtained from training the folds. Ignore the best iterations for GNB.
    _, threshold = hf_get_best_iterations_and_threshold(
        tier, ir, bootstrap, aug, files_expected, model_name)
        
    # Train the classifier using tuned hyperparameters read from a JSON file.
    # Instead of early stopping, disable and specify n_estimators as the median of best iterations
    # for fold training.
    n_neg = (y_train_scenario_aug == 0).sum()     # Majority class - except for high minority augmentation
    n_pos = (y_train_scenario_aug == 1).sum()     # Minority class

    # Train GaussianNB model
    fit_start = perf_counter()
    scenario_model = GaussianNB()
    scenario_model.fit(X_train_scenario_imp_aug, y_train_scenario_aug)
    fit_runtime = (perf_counter() - fit_start)/60  # Timer end

    # Evaluate the model against the test set using the best threshold from fold training
    #print('y_test:', y_test.shape)
    results, y_test_proba = hf_evaluate_scenario_results(
        scenario_model, threshold, X_test_imp, y_test, ir)

    # Save the scenario metadata
    results['n_estimators'] = 0                      # NA for GaussianNB
    results['n_train'] = y_train_scenario.shape[0]
    results['nominal_ir'] = ir
    results['augmentation_pct'] = aug
    results['n_neg_real'] = n_neg_real.item()
    results['n_pos_real'] = n_pos_real.item()
    results['n_synthetic'] = y_synthetic.shape[0] if augmented else 0
    results['n_train_aug'] = y_train_scenario_aug.shape[0]
    results['n_train_aug_neg'] = n_neg.item()
    results['n_train_aug_pos'] = n_pos.item()
    results['augmented_ir'] = (n_neg/n_pos).item() if augmented else ir
    results['fit_runtime'] = fit_runtime

    # Log the results - Pass fold=-1 to indicate a scenario instead of fold
    hf_save_results(tier, ir, bootstrap, -1, aug, 
                    results, y_test, y_test_proba, 
                    model_name)
    return results

### Wrapper function for GNB fold training (`hf_fold_training_gnb`)

In [82]:
import os
from time import perf_counter

def hf_fold_training_gnb(tier, ir, augs, n_folds=5, testmode=False):
    '''
    Wrapper function to call `train_and_evaluate_gnb_fold`.
    '''
    # Call the fold training function and display progress by fold
    start = perf_counter()
    print(f'Train GNB and evaluate ({tier:6}, {ir:03d}):\n  ', end='')
    
    for aug in augs:
        # Check for the stop-signal
        if os.path.exists('..\\STOP.txt'):
            print("Stop signal detected. Model training halted.")
        else:
            # Do not apply augmentation to balanced scenarios
            if not((ir==1) and (aug>0)):
                print(f'Aug{aug:d}', end='')
                for fold in range(n_folds):
                    print('.', end='')
                    if not testmode:
                        train_and_evaluate_gnb_fold(tier, ir, 0, fold, aug)
                        pass            
                else:
                    print(' ', end='')
            else:
                # Do not call `train_and_evaluate_gnb_fold` if IR=1 and aug>0
                pass
    else:
        print(f'  | GNB Scenario runtime: {(perf_counter() - start) / 60:5.2f} minutes')
        if testmode:
            print(' ** TEST MODE MESSAGE: GNB Training was not completed **')

### Wrapper function for GNB scenario training (`hf_scenario_training_gnb`)

In [83]:
import os
from time import perf_counter

def hf_scenario_training_gnb(tier, ir, augs, testmode=False):
    '''
    Train and scenarios with and without augmentation. Scenario are evaluated against a
    holdout test set instead of using cross validation.
    '''
    start = perf_counter()
    print(f'Train and evaluate GNB ({tier:6}, {ir:03d}):\n  ', end='')
    for aug in augs:
        # Check for the stop-signal
        if os.path.exists('..\\STOP.txt'):
            print("Stop signal detected. Model training halted.")
        else:
            # Skip augmented training if scenario is balanced
            if not((ir == 1) and (aug != 0)):
                print(f'Aug{aug:d}', end='')
                if not testmode:
                    pass
                    #print('Calling train_and_evaluate_gnb_scenario...')
                    train_and_evaluate_gnb_scenario(tier, ir, 0, aug)
                print('. ', end='')
    else:
        print(f'  | Scenario runtime: {(perf_counter() - start)/60:5.1f} minutes')
        if testmode:
            print('TESTMODE MESSAGE: Training was not completed.')

### Combined fold-scenario training wrapper - GNB

In [86]:
from time import time

def hf_fold_scenario_training_gnb(scenarios):
    '''
    Call the GNB fold and scenario training wrapper functions for scenarios.

    Parameters
    ----------
    scenarios : dict like {'tier': {'ir': [], 'augs': []}}
    '''
    start = perf_counter()
    
    # Iterate through the tiers, IRs, and augs, and call fold-level and scenario-level training
    # functions
    #for tiers, irs, augs in scenarios:
    for tier, params in scenarios.items():
        augs = params['augs']
    
        for ir in params['irs']:
            print(f'Gaussian naive Bayes training {tier} tier IR {ir} with augs {augs}...')
            hf_fold_training_gnb(tier, ir, augs, testmode=False)
            hf_scenario_training_gnb(tier, ir, augs, testmode=False)
    
    print(f'\nRuntime: {(perf_counter() - start) / 60:4.1f} minutes')
            
    print('\n-- All scenarios complete --')

## Gaussian naive Bayes training for all scenarios

All scenario lists to be trained:
```
hf_fold_scenario_training_gnb(scenarios_main)
hf_fold_scenario_training_gnb(scenarios_fine_grained)
hf_fold_scenario_training_gnb(scenarios_augmentation_800)
hf_fold_scenario_training_gnb(scenarios_real)
hf_fold_scenario_training_gnb(scenarios_extra_small)
```

### 1. Main scenarios IR 1-243, augmenation 0-400%

In [89]:
hf_fold_scenario_training_gnb(scenarios_main)

Gaussian naive Bayes training large tier IR 3 with augs [0, 50, 100, 200, 400]...
Train GNB and evaluate (large , 003):
  Aug0..... Aug50..... Aug100..... Aug200..... Aug400.....   | GNB Scenario runtime:  1.61 minutes
Train and evaluate GNB (large , 003):
  Aug0. Aug50. Aug100. Aug200. Aug400.   | Scenario runtime:   0.3 minutes
Gaussian naive Bayes training large tier IR 9 with augs [0, 50, 100, 200, 400]...
Train GNB and evaluate (large , 009):
  Aug0..... Aug50..... Aug100..... Aug200..... Aug400.....   | GNB Scenario runtime:  1.46 minutes
Train and evaluate GNB (large , 009):
  Aug0. Aug50. Aug100. Aug200. Aug400.   | Scenario runtime:   0.3 minutes
Gaussian naive Bayes training large tier IR 81 with augs [0, 50, 100, 200, 400]...
Train GNB and evaluate (large , 081):
  Aug0..... Aug50..... Aug100..... Aug200..... Aug400.....   | GNB Scenario runtime:  1.43 minutes
Train and evaluate GNB (large , 081):
  Aug0. Aug50. Aug100. Aug200. Aug400.   | Scenario runtime:   0.3 minutes
Gau

### 2. Fine-grained augmentation interval 0-50%

In [93]:
hf_fold_scenario_training_gnb(scenarios_fine_grained)

Gaussian naive Bayes training medium tier IR 3 with augs [4, 8, 13, 18, 22, 28, 33, 38, 44]...
Train GNB and evaluate (medium, 003):
  Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44.....   | GNB Scenario runtime:  1.39 minutes
Train and evaluate GNB (medium, 003):
  Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44.   | Scenario runtime:   0.3 minutes
Gaussian naive Bayes training medium tier IR 9 with augs [4, 8, 13, 18, 22, 28, 33, 38, 44]...
Train GNB and evaluate (medium, 009):
  Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44.....   | GNB Scenario runtime:  1.40 minutes
Train and evaluate GNB (medium, 009):
  Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44.   | Scenario runtime:   0.3 minutes
Gaussian naive Bayes training medium tier IR 27 with augs [4, 8, 13, 18, 22, 28, 33, 38, 44]...
Train GNB and evaluate (medium, 027):
  Aug4..... Aug8..... Aug13..... Aug18..... Aug2

### 3. 800% Augmentation

In [88]:
hf_fold_scenario_training_gnb(scenarios_augmentation_800)

Gaussian naive Bayes training medium tier IR 243 with augs [800]...
Train GNB and evaluate (medium, 243):
  Aug800.....   | GNB Scenario runtime:  0.18 minutes
Train and evaluate GNB (medium, 243):
  Aug800.   | Scenario runtime:   0.0 minutes
Gaussian naive Bayes training medium tier IR 729 with augs [800]...
Train GNB and evaluate (medium, 729):
  Aug800.....   | GNB Scenario runtime:  0.17 minutes
Train and evaluate GNB (medium, 729):
  Aug800.   | Scenario runtime:   0.0 minutes
Gaussian naive Bayes training small tier IR 243 with augs [800]...
Train GNB and evaluate (small , 243):
  Aug800.....   | GNB Scenario runtime:  0.12 minutes
Train and evaluate GNB (small , 243):
  Aug800.   | Scenario runtime:   0.0 minutes

Runtime:  0.6 minutes

-- All scenarios complete --


### 4. Real data augmentation

In [94]:
hf_fold_scenario_training_gnb(scenarios_real)

Gaussian naive Bayes training smallaug tier IR 1 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]...
Train GNB and evaluate (smallaug, 001):
  Aug0.....   | GNB Scenario runtime:  0.11 minutes
Train and evaluate GNB (smallaug, 001):
  Aug0.   | Scenario runtime:   0.0 minutes
Gaussian naive Bayes training smallaug tier IR 3 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]...
Train GNB and evaluate (smallaug, 003):
  Aug0..... Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44..... Aug50..... Aug100..... Aug200.....   | GNB Scenario runtime:  1.50 minutes
Train and evaluate GNB (smallaug, 003):
  Aug0. Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44. Aug50. Aug100. Aug200.   | Scenario runtime:   0.3 minutes
Gaussian naive Bayes training smallaug tier IR 9 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]...
Train GNB and evaluate (smallaug, 009):
  Aug0..... Aug4..... Aug8..... Aug13..... Aug18..

### 5. Extra-small scenarios

In [106]:
# GNB cannot support all IRs for extra small scenarios
# Very small and extra small scenarios
scenarios_extra_small = {
    'smalla': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'smallb': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'smallc': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'smalld': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'xsmalla': {'irs': [1, 3, 9, 27], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    'xsmallb': {'irs': [1, 3, 9, 27], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    # C cannot support IR 27 or greater
    'xsmallc': {'irs': [1, 3, 9], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]},
    # D cannot support IR 9 or greater
    'xsmalld': {'irs': [1, 3], 'augs': [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]}
}

In [104]:
hf_fold_scenario_training_gnb(scenarios_extra_small)

Gaussian naive Bayes training smalla tier IR 1 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]...
Train GNB and evaluate (smalla, 001):
  Aug0.....   | GNB Scenario runtime:  0.08 minutes
Train and evaluate GNB (smalla, 001):
  Aug0.   | Scenario runtime:   0.0 minutes
Gaussian naive Bayes training smalla tier IR 3 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]...
Train GNB and evaluate (smalla, 003):
  Aug0..... Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44..... Aug50.....   | GNB Scenario runtime:  0.96 minutes
Train and evaluate GNB (smalla, 003):
  Aug0. Aug4. Aug8. Aug13. Aug18. Aug22. Aug28. Aug33. Aug38. Aug44. Aug50.   | Scenario runtime:   0.2 minutes
Gaussian naive Bayes training smalla tier IR 9 with augs [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]...
Train GNB and evaluate (smalla, 009):
  Aug0..... Aug4..... Aug8..... Aug13..... Aug18..... Aug22..... Aug28..... Aug33..... Aug38..... Aug44..... Aug50.....   | GNB Scenario

## Aggregate Evaluation Results

This section aggregates the fold and scenario-level training results and calculates F1 recovery for all models. Results are combined into three JSON files:
```
..
    |-- results
        |-- xgb-fold-results-b00.json         # XGB fold evaluation results for all scenarios 
        |-- xgb-scenarios-results-b00.json    # XGB scenario evaluation results for all scenarios
        |-- gnb-fold-results-b00.json         # GNB fold evaluation results for all scenarios
        |-- gnb-scenario-results-b00.json     # GNB scenario evaluation results for all scenarios
```

### Utility to check which scenarios have been calculated

In [127]:
from pathlib import Path

# Iterate through all the scenarios for one bootstrap
def hf_find_results(bootstrap, tabulate=False):

    # Print the number of results files for each augmentation level. Folds have k, scenarios have 1
    def show_scenario_results(results):
        for (tier, ir), aug_counts in results.items():
            print(tier, ir, ':', end='')
            if len(aug_counts) == 0:
                pass
            else:
                for aug, count in aug_counts.items():
                    print(f'{aug}: {count:1d} ', end='')
            print()
            
    # Get a list of scenarios for the bootstrap
    root = Path(f'..\\scenarios')
    scenario_folders = [f for f in root.iterdir() if f.is_dir() and f.match(f'*b{bootstrap:02d}')]
    
    # Iterate through the fold and scenario-level results for each model
    model_paths = {'xgb': {'fold': 'fold-results', 'scenario': 'scenario-results'},
                   'gnb': {'fold': 'fold-results-gnb', 'scenario': 'scenario-results-gnb'}}
    all_models = {'xgb': {}, 'gnb': {}}
    for model, levels in model_paths.items():
        
        fold_folder = levels['fold']
        scenario_folder = levels['scenario']
        fold_results = {}
        scenario_results = {}

        # Iterate through the scenarios
        for s in scenario_folders:
            tier, ir, aug = s.name.split('-')

            # Search for fold results
            folds = {}
            path = s / fold_folder

            # Get a list of files in the folder then count the number of folds for each 
            # augmentation level, then add the total to fold-level summary
            files = path.glob('results*.json')

            # Iterate through the files in the folder
            for f in files:
                _, fold, aug = f.name.split('-')       # Parse the file name to get aug%
                aug = aug.split('.')[0]                # Strip off the '.json'
                folds[aug] = folds.get(aug, 0) + 1     # Increment the aug count
            fold_results[(tier, ir)] = folds
            all_models[model]['fold'] = fold_results

            # Search for scenario results
            scenarios = {}
            path = s / scenario_folder
            
            # Scenario-level models are run only once per scenario using hold out test set
            files = path.glob('results*.json')
            for f in files:
                _, aug = f.name.split('-')               # Parse the file name to get aug%
                aug = aug.split('.')[0]                  # Strip off the '.json'
                scenarios[aug] = 1                       # Every aug level found will be 1
            scenario_results[(tier, ir)] = scenarios
            all_models[model]['scenario'] = scenario_results

        else:
            if tabulate:
                print(f'\nFOLD-LEVEL RESULTS - {model}\n')
                show_scenario_results(fold_results)
                print(f'\nSCENARIO-LEVEL RESULTS - {model}\n')
                show_scenario_results(scenario_results)
      
# Call the function for bootstrap 0
_ = hf_find_results(0, True)


FOLD-LEVEL RESULTS - xgb

large ir001 :aug000: 5 
large ir003 :aug000: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
large ir009 :aug000: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
large ir027 :aug000: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
large ir081 :aug000: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
large ir243 :aug000: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
medium ir001 :aug000: 5 
medium ir003 :aug000: 5 aug004: 5 aug008: 5 aug013: 5 aug018: 5 aug022: 5 aug028: 5 aug033: 5 aug038: 5 aug044: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
medium ir009 :aug000: 5 aug004: 5 aug008: 5 aug013: 5 aug018: 5 aug022: 5 aug028: 5 aug033: 5 aug038: 5 aug044: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
medium ir027 :aug000: 5 aug004: 5 aug008: 5 aug013: 5 aug018: 5 aug022: 5 aug028: 5 aug033: 5 aug038: 5 aug044: 5 aug050: 5 aug100: 5 aug200: 5 aug400: 5 
medium ir081 :aug000: 5 aug004: 5 aug008: 5 aug013: 5 aug018: 5 aug022: 5 aug028: 5 aug033: 5 aug038: 5 aug044: 5 aug050: 5 aug100: 5 aug20

### Read and aggregate fold level results (`hf_read_fold_results`)

Aggregate the results from the multiple folds into a single set by taking the median of each metric across the folds.

In [128]:
def hf_read_fold_results(tier, ir, bootstrap, aug, files_expected=5, model_name='xgb'):
    '''
    Read and aggregate the results for all folds in a scenario

    NOTE: Files expected will normally equal the number of folds used for evaluation.
    '''
    # Search the scenario results folder for fold-level results and check that the expected
    # number of files were found
    root = Path('..\\scenarios\\')
    if model_name == 'xgb':
        results_path = root / f'{tier}-ir{ir:03}-b{bootstrap:02}' / 'fold-results'
    else:
        results_path = root / f'{tier}-ir{ir:03}-b{bootstrap:02}' / f'fold-results-{model_name}'
        
    all_files = os.listdir(results_path)
    files = [f for f in all_files if f'{aug:03}.json' in f]
    files_found = len(files)
    #print('searching:', results_path, 'for', f'{aug:03d}')
    #print('all_file:', all_files)
    #print('aug files:', files)
    
    assert files_found == files_expected, \
    f'Mismatch: expected {files_expected} results files, found {files_found}' \
    f' for tier {tier} IR {ir} and aug {aug}'

    # Read the results file for each fold
    fold_results = []
    for file in files:
        with open(results_path / file, 'r') as f:
            results = json.load(f)
        fold_results.append(results)
        
    # Calculate the median and standard deviation of results for every metric over the folds.
    # For simplicity, calculate the median of every value even though median is not relvevant 
    # for all.
    fold_results_t, fold_stats = {}, {}
    for metric in fold_results[0].keys():
        metrics= [r[metric] for r in fold_results]
        fold_stats[metric + '_median'] = np.median(metrics).item()
        fold_stats[metric + '_std'] = np.std(metrics).item()

    # Return a dict of median and standard deviation of fold level results
    return fold_stats

In [129]:
#TEST
print(hf_read_fold_results('medium', 81, 0, 100, 5))

{'n_estimators_median': 800.0, 'n_estimators_std': 0.0, 'early_stopping_rounds_median': 100.0, 'early_stopping_rounds_std': 0.0, 'best_iteration_median': 94.0, 'best_iteration_std': 278.82797564089583, 'pr_auc_median': 0.03561640668643694, 'pr_auc_std': 0.011107073819194695, 'roc_auc_median': 0.7114592113646309, 'roc_auc_std': 0.02455467740767077, 'log_loss_median': 0.20541173756590284, 'log_loss_std': 0.044292717666085384, 'threshold_tuning_runtime_median': 0.01346520833321847, 'threshold_tuning_runtime_std': 0.0030697866984431546, 'best_threshold_median': 0.48, 'best_threshold_std': 0.059799665550904206, 'best_f1_median': 0.08290155440414508, 'best_f1_std': 0.023269397483832846, 'best_rec_median': 0.10091743119266056, 'best_rec_std': 0.03977886860308037, 'best_pr_median': 0.09523809523809523, 'best_pr_std': 0.05958036648532477, 'n_train_median': 35747.0, 'n_train_std': 0.4, 'nominal_ir_median': 81.0, 'nominal_ir_std': 0.0, 'augmentation_pct_median': 100.0, 'augmentation_pct_std': 0.0

In [ ]:
# TEST - Using experimentatal data

# Train and imbalanced scenarios with and without augmentation
for aug in [0, 50]:
    for fold in range(5):
        #print('Train and evaluate:', 'small', 27, 'fold', fold, 'aug', aug, ' ...')
        #train_and_evaluate_fold('small', 27, 0, fold, aug)
        pass
       
# Train balanced scenarios
for fold in range(5):    
    #print('Train and evaluate:', 'small', 1, 'fold', fold, 'aug', 0, ' ...')
    #train_and_evaluate_fold('small', 1, 0, fold, 0)
    pass

print('\n-- All folds comlpete --')

### Work-around for large scenario

The large scenaro has insufficient minority samples to construct a balanced scearnio. IR 3 is the closest to a balanced scenario available. Therefore, to aprpoximate F1 recovery calculations, copy the IR 3 results to the IR 1. Manually assign F1 values from the medium tier to the results for Gaussian naive Bayes and XGBoost so that nominal F1 calculations can be completed. This is justified because the results show that sample size has a small effect on F1, so the values will be only slightly lower than expected.

In [134]:
# NOTE To apply to GNB, add `-gnb` suffix to fold and scenario results paths

bootstrap = 0
model = 'gnb'                       # Set to 'gnb' or 'xgb'

if model == 'gnb':
    # Set the fold input-output file paths
    ir1_path = f'..\\scenarios\\large-ir001-b{bootstrap:02d}\\fold-results-gnb\\'
    ir3_path = f'..\\scenarios\\large-ir003-b{bootstrap:02d}\\fold-results-gnb\\'
    
    # Hard code the replacement values
    f1_increment = 0.0
    f1_loss_approximation = 0.0     # Apply if prediction is not availble to avoid division by zero
    f1_approximation_w = 0.667      # This is the predicted value discovered through experimentation
                                    # using GaussianNB    
    f1_approximation_u = 0.1600     # GaussianNB - unweighted
else:
    # Set the fold input-output file paths
    ir1_path = f'..\\scenarios\\large-ir001-b{bootstrap:02d}\\fold-results\\'
    ir3_path = f'..\\scenarios\\large-ir003-b{bootstrap:02d}\\fold-results\\'
    
    # Hard code the replacement values
    f1_increment = 0.0
    f1_loss_approximation = 0.0      # Apply if prediction is not availble to avoid division by zero
    
    f1_approximation_w = 0.717       # This is the predicted value discovered through experimentation
                                     # using XGBoost
    f1_approximation_u = 0.2709      # XGBoost - unweighted

# Update fold results
for fold in range(5):
    # Create the destination path and file name
    filename = f'results-fold{fold}-aug000.json'

    # Open the IR 3 file. Add a small increment to F1 to avoid division-by-zero error when
    # calcualting recovery
    print('Source:', ir3_path + filename)
    with open(ir3_path + filename, 'r') as f:
        ir3_results = json.load(f)
    ir3_results['best_f1'] += f1_increment

    # Add an approximate F1 loss from IR 1 to IR 3
    ir3_results['best_f1'] += f1_loss_approximation

    # Override with approximation based on experimental results if available
    ir3_results['best_f1'] = f1_approximation_w

    # Update IR from 3 to 1
    ir3_results['nominal_ir'] = 1
    ir3_results['augmented_ir'] = 1
            
    # Save the IR 3 results to IR 1
    print(f'Copying IR 3 fold {fold} to', ir1_path + filename)
    with open(ir1_path + filename, 'w') as f:
        json.dump(ir3_results, f, indent=4)

# Repeat for the scenario level results

# Create the destination path and file name
if model == 'gnb':
    ir1_path = f'..\\scenarios\\large-ir001-b{bootstrap:02d}\\scenario-results-gnb\\'
    ir3_path = f'..\\scenarios\\large-ir003-b{bootstrap:02d}\\scenario-results-gnb\\'
else:
    ir1_path = f'..\\scenarios\\large-ir001-b{bootstrap:02d}\\scenario-results\\'
    ir3_path = f'..\\scenarios\\large-ir003-b{bootstrap:02d}\\scenario-results\\'
    
filename = f'results-aug000.json'

# Open the IR 3 file. Add a small increment to F1 to avoid division-by-zero error when
# calcualting recovery
print('Source:', ir3_path + filename)
with open(ir3_path + filename, 'r') as f:
    ir3_results = json.load(f)
ir3_results['f1_score'] += f1_increment
ir3_results['f1_score_w'] += f1_increment

# Add an approximate F1 loss from IR 1 to IR 3
ir3_results['f1_score'] += f1_loss_approximation
ir3_results['f1_score_w'] += f1_loss_approximation

# Override with approximation based on experimental results if known
if f1_approximation_w != 0:
    ir3_results['f1_score'] = f1_approximation_u
    ir3_results['f1_score_w'] = f1_approximation_w

# Update IR from 3 to 1
ir3_results['nominal_ir'] = 1
ir3_results['augmented_ir'] = 1
    
# Save the IR 3 results to IR 1
print(f'\nCopying IR 3 to', ir1_path + filename)
with open(ir1_path + filename, 'w') as f:
    json.dump(ir3_results, f, indent=4)        

Source: ..\scenarios\large-ir003-b00\fold-results-gnb\results-fold0-aug000.json
Copying IR 3 fold 0 to ..\scenarios\large-ir001-b00\fold-results-gnb\results-fold0-aug000.json
Source: ..\scenarios\large-ir003-b00\fold-results-gnb\results-fold1-aug000.json
Copying IR 3 fold 1 to ..\scenarios\large-ir001-b00\fold-results-gnb\results-fold1-aug000.json
Source: ..\scenarios\large-ir003-b00\fold-results-gnb\results-fold2-aug000.json
Copying IR 3 fold 2 to ..\scenarios\large-ir001-b00\fold-results-gnb\results-fold2-aug000.json
Source: ..\scenarios\large-ir003-b00\fold-results-gnb\results-fold3-aug000.json
Copying IR 3 fold 3 to ..\scenarios\large-ir001-b00\fold-results-gnb\results-fold3-aug000.json
Source: ..\scenarios\large-ir003-b00\fold-results-gnb\results-fold4-aug000.json
Copying IR 3 fold 4 to ..\scenarios\large-ir001-b00\fold-results-gnb\results-fold4-aug000.json
Source: ..\scenarios\large-ir003-b00\scenario-results-gnb\results-aug000.json

Copying IR 3 to ..\scenarios\large-ir001-b00\s

### Fold level recovery calculation (`f1_recovery_fold`)

Calculate F1 recovery for aggreged fold evaluation results. Recovery is the amount of performance that augmentation achieved compared to the unaugmented scenario, as a proportion of the performance that was induced by imbalance compared to the balanced scenario. The calculation requires three scenario results:
* balanced scenario
* imbalanced scenario without augmentation
* imbalanced scenario with augmentation

In [ ]:
import json

def f1_recovery_fold(tier, irs, augs, bootstrap, model_name):

    '''
    Iterate through the fold-level results, aggregate, calculate recovery, then combine into a 
    single dict.

    NOTES
    -----
    IR 1 must be included for each tier.
    0% augmentation must be included for each IR.
    '''
    all_results = {}

    #for tier, ir, aug in hf_iter_scenarios(include_large_ir1=True, include_phase2=True):
    for ir in irs:
        print(f'  IR{ir:03d}', end='')
        for aug in augs:

            # Skip IR 1 with >0% augmentation
            if (ir==1) and (aug>0):
                continue
       
            # Call helper function to read and aggregated results
            results = hf_read_fold_results(tier, ir, 0, aug, 5, model_name)
            balanced = f'{tier}-ir{1:03d}-aug{0:03d}'
            baseline = f'{tier}-ir{ir:03d}-aug{0:03d}'
            scenario = f'{tier}-ir{ir:03d}-aug{aug:03d}'
        
            # F1 Recovery calculations
            #   Balanced F1 = F1 for for the balanced scenario (IR=1)
            #   Baseline F1 = F1 for the imbalanced scenario with 0% augmentation
            #   Augmented F1 = F1 for the imbalanced, augmented scenario
            #   F1 Recovery = (Augmented F1 - Baseline F1) / (Balanced F1 - Baseline F1)
            #
            # Scenarios are calculated in sequence: balanced > unbalanced 0% aug > unbalanced augmented
            if ir == 1:
                results['f1_balanced'] = results['best_f1_median']
                results['f1_baseline'] = None
                results['f1_loss'] = None
                results['f1_augmented'] = None
                results['f1_recovered'] = None
                results['f1_recovery_proportion'] = None
                results['f1_recovery_percentage'] = None
                
            else:
                results['f1_balanced'] = all_results[balanced]['f1_balanced']
                if aug==0:
                    results['f1_baseline'] = results['best_f1_median']
                    results['f1_loss'] = results['f1_balanced'] - results['f1_baseline']
                    results['f1_augmented'] = None
                    results['f1_recovered'] = None
                    results['f1_recovery_proportion'] = None
                    results['f1_recovery_percentage'] = None
    
                else:
                    #results['f1_baseline'] = all_results[(tier, ir, 0)]['f1_baseline']
                    results['f1_baseline'] = all_results[baseline]['f1_baseline']
                    #results['f1_loss'] = all_results[(tier, ir, 0)]['f1_loss']
                    results['f1_loss'] = all_results[baseline]['f1_loss']
                    results['f1_augmented'] = results['best_f1_median']
        
                    # Calculated F1 recovery
                    results['f1_recovered'] = results['f1_augmented'] - results['f1_baseline']
                    results['f1_recovery_proportion'] = (results['f1_recovered'] / 
                                                         results['f1_loss'])
                    results['f1_recovery_percentage'] = results['f1_recovery_proportion'] * 100
    
            # Add the results to the fold-level dict
            all_results[scenario] = results
        else:
            pass
            #print(f'  Adding fold-level results for {tier} IR {ir} bootstrap {bootstrap}')
    else:
        print('  Complete')
        return all_results

### Scenario-level recovery calculation (`f1_recovery_scenario`)

Calculate F1 recovery from scenario (test set) evaluation results.

In [136]:
def f1_recovery_scenario(tier, irs, augs, bootstrap, model_name='xgb'):
    '''
    Read the scenario-level results for one tier, calculate recovery, then combine into a 
    single dict.
    '''
    all_results = {}

    #for tier, ir, aug in hf_iter_scenarios(include_large_ir1=True, include_phase2=True):
    for ir in irs:
        print(f'  IR{ir:03d}', end='')
        for aug in augs:

            # Skip IR 1 with >0% augmentation
            if (ir==1) and (aug>0):
                continue
        
            # Define the balanced, baseline, and current scenarios indices
            balanced = f'{tier}-ir{1:03d}-aug{0:03d}'    # IR=1, 0% augmentation
            baseline = f'{tier}-ir{ir:03d}-aug{0:03d}'   # IR=ir, 0% augmentation
            scenario = f'{tier}-ir{ir:03d}-aug{aug:03d}' # IR=ir, augmentation=aug
    
            # Read the scenario results for the tier-ir-aug
            #path = f'..\\scenarios\\{tier}-ir{ir:03d}-b{bootstrap:02d}\\scenario-results\\'
            path = f'..\\scenarios\\{tier}-ir{ir:03d}-b{bootstrap:02d}\\'
            if model_name == 'xgb':
                path = f'{path}scenario-results\\'
            else:
                path = f'{path}scenario-results-{model_name}\\'
            
            filename = f'results-aug{aug:03d}.json'
            with open(path + filename, 'r') as f:
                results = json.load(f)
                
            # Scenarios are calculated in sequence: balanced > unbalanced 0% aug > unbalanced augmented
            # Calculate recovery based on stanard metrics - calculated against the unweight test set - 
            # and the weighted metrics - calculated against the weighted test set
            if ir == 1:
                results['f1_balanced'] = results['f1_score']
                results['f1_balanced_w'] = results['f1_score_w']
                results['f1_baseline'] = None
                results['f1_baseline_w'] = None     
                results['f1_loss'] = None
                results['f1_loss_w'] = None
                results['f1_augmented'] = None
                results['f1_augmented_w'] = None
                results['f1_recovered'] = None
                results['f1_recovered_w'] = None
                results['f1_recovery_proportion'] = None
                results['f1_recovery_proportion_w'] = None
                results['f1_recovery_percentage'] = None
                results['f1_recovery_percentage_w'] = None
    
            else:
                results['f1_balanced'] = all_results[balanced]['f1_balanced']
                results['f1_balanced_w'] = all_results[balanced]['f1_balanced_w']
                if aug==0:
                    results['f1_baseline'] = results['f1_score']
                    results['f1_baseline_w'] = results['f1_score_w']
                    results['f1_loss'] = results['f1_balanced'] - results['f1_baseline']
                    results['f1_loss_w'] = results['f1_balanced_w'] - results['f1_baseline_w']
                    results['f1_augmented'] = None
                    results['f1_augmented_w'] = None
                    results['f1_recovered'] = None
                    results['f1_recovered_w'] = None
                    results['f1_recovery_proportion'] = None
                    results['f1_recovery_proportion_w'] = None
                    results['f1_recovery_percentage'] = None
                    results['f1_recovery_percentage_w'] = None
    
                else:
                    results.update({
                        'f1_baseline': all_results[baseline]['f1_baseline'],
                        'f1_baseline_w': all_results[baseline]['f1_baseline_w'],
                        'f1_loss': all_results[baseline]['f1_loss'],
                        'f1_loss_w': all_results[baseline]['f1_loss_w'],
                        'f1_augmented': results['f1_score'],
                        'f1_augmented_w': results['f1_score_w']
                    })
                    results.update({
                        # Calculate F1 recovery for augmented datasets
                        'f1_recovered': results['f1_augmented'] - results['f1_baseline'],
                        'f1_recovered_w': results['f1_augmented_w'] - results['f1_baseline_w']
                    })
                    results.update({
                        'f1_recovery_proportion': (results['f1_recovered']/results['f1_loss']),
                        'f1_recovery_proportion_w':(results['f1_recovered_w']/results['f1_loss_w'])
                    })
                    results.update({
                        'f1_recovery_percentage': results['f1_recovery_proportion'] * 100,
                        'f1_recovery_percentage_w': results['f1_recovery_proportion_w'] * 100
                    })            
                    
            # Add the updated results to the fold-level dict
            all_results[scenario] = results
        else:
            pass
            #print(f'  Adding scenario-level results for {tier} IR {ir} bootstrap {bootstrap}')
    else:
        print('  Complete')
        return all_results

### Wrapper function for fold and scenario recovery (`hf_save_f1_recovery_model`)

Wrapper function which call F1 recovery functions for fold- and scenario-level evaluation results. `model_name` can be passed as a parameter to specify results from a particular model e.g. `xgb` for XGBoost or `gnb` for Gaussian Naive Bayes.

In [137]:
import os, json

def hf_save_f1_recovery_model(stages, bootstrap, model_name):
    '''
    Iterate through all aggregated fold and scenarios results, calculate F1 recovery and merge 
    into two files; one for all aggregated fold results and one for scenario results. 

    Notes
    -----
    Each call to `f1_recovery_fold` and `f1_recovery_scenario` returns a dict of all results for
    one tier of aggregated fold or scenario results.
    '''
    fold_results = {}
    scenario_results = {}
        
    # Add the fold-level results for each scenario to the complete result set
    print('\nFOLD LEVEL RESULTS')
    for tiers, irs, augs in stages:     

        # Make one call to `f1_recovery_fold` per size tier
        for tier in tiers:
            print(f'Adding tier {tier}')
            fold_results.update(f1_recovery_fold(tier, irs, augs, bootstrap, model_name))

    # Add the scenario-level results for each scenario to the complete result set
    print('\nSCENARIO LEVEL RESULTS')
    for tiers, irs, augs in stages:     

        # Make one call to `f1_recovery_scenario` per size tier
        for tier in tiers:
            print(f'Adding tier {tier}')
            scenario_results.update(f1_recovery_scenario(tier, irs, augs, bootstrap, model_name))
            #scenario_results.update({tier: 'test'})

    # Save the fold-level and scenario-level complete results to JSON. Ensure the path exists
    path = f'..\\results\\'
    os.makedirs(path, exist_ok=True)

    # Fold-level
    #    if model_name == 'xgb':
    #        filename = f'fold-results-b{bootstrap:02d}.json'
    #   else:
    #       filename = f'{model_name}-fold-results-b{bootstrap:02d}.json'
    #   
    filename = f'{model_name}-fold-results-b{bootstrap:02d}.json'

    print(f'\nSaving all fold results to {path}{filename}')
    with open(path + filename, 'w') as f:
        json.dump(fold_results, f, indent=2)


    # Scenario-level
    #if model_name =='xgb':
    #    filename = f'scenario-results-b{bootstrap:02d}.json'
    #else:
    #    filename = f'{model_name}-scenario-results-b{bootstrap:02d}.json'
    filename = f'{model_name}-scenario-results-b{bootstrap:02d}.json'

    print(f'\nSaving all scenario results to {path}{filename}')
    with open(path + filename, 'w') as f:
        json.dump(scenario_results, f, indent=2)        

### Create scenario dicts for XGBoost and Gaussian Naive Bayes

In [144]:
'''
Define the tier, IR, augmentation combinations used for each experimental stage
  [([<tier list>], [<IR list>], [<augmentation level list>]), ...]

NOTES
-----
1. Each tier (e.g. large) must include the balanced scenario in the IR list.
I.e. ([1, ...], [...], [...]).
2. Each tier must include the baseline scenario of augmentation = 0% 
I.e. ([...]. [...], [0, ...])
3. Tiers can be repeated without being duplicating in the saved results provided that the 
balanced IR and basline augmentation are included in the IR and augmentation lists, even if they 
have been included for the same tier in another stage. E.g. 
  (['medium'], [1, 3], ...), (['medium'], [1, 243], ...) is allowed. 
  (['medium'], [1, 3], ...), (['medium'], [243], ...) is not allowed.
'''

# XGBoost
xgb_stages = [
    
    # Stage 1 - Main scenarios
    (['large'], [1, 3, 9, 27, 81, 243], [0, 50, 100, 200, 400]),
    (['medium', 'small'], [1, 3, 9, 27, 81, 243], [0, 50, 100, 200, 400]),
    
    # Stage 2 - Fine grained augmentation 0-50%
    (['medium'], [1, 3, 9, 27, 81], [0, 4, 8, 13, 18, 22, 28, 33, 38, 44]),
    
    # 3. 800% Augmentation
    (['medium'], [1, 243, 729], [0, 50, 100, 200, 400, 800]),
    (['small'], [1, 243], [0, 50, 100, 200, 400, 800]),

    
    # 4. Augment using real data
    (['smallaug'], [1, 3, 9, 27, 81], [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]),

    # Stage 5.1 - Smaller tier & fewer IRs + fine grained augmentation
    (['smalla', 'smallb', 'smallc', 'smalld'], 
     [1, 3, 9], 
     [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]),

    # Stage 5.2 - Extra small tiers + fine grained augmentation
    (['xsmalla', 'xsmallb', 'xsmallc', 'xsmalld'], 
     [1, 3, 9, 27], 
     [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]),
]

# Gaussian Naive Bayes
gnb_stages = [
    
    # Phase 1 - Course grid
    (['large'], [1, 3, 9, 27, 81, 243], [0, 50, 100, 200, 400]),
    (['medium', 'small'], [1, 3, 9, 27, 81], [0, 50, 100, 200, 400]),
    
    # Phase 5 - Use real data for augmentation
    (['smallaug'], [1, 3, 9, 27, 81], [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]),
    
    # Phase 2 - Medium + fine grained augmentation
    (['medium'], [1, 3, 9, 27, 81], [0, 4, 8, 13, 18, 22, 28, 33, 38, 44]),
    
    # Phase 3 - Smaller tier & fewer IRs + fine grained augmentation
    (['smalla', 'smallb', 'smallc', 'smalld'], 
     [1, 3, 9], 
     [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]),
    
    # Phase 4 - Extra small tiers + fine grained augmentation
    (['xsmalla', 'xsmallb', 'xsmallc', 'xsmalld'], 
     [1, 3], 
     [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50]),

    # Diagnostic - extreme augmentation
    (['medium'], [1, 243, 729], [0, 50, 100, 200, 400, 800]),
    (['small'], [1, 243], [0, 50, 100, 200, 400, 800])
]

### Save results for XGBoost

In [143]:
# Call the wrapper function for XGB
hf_save_f1_recovery_model(xgb_stages, 0, 'xgb')


FOLD LEVEL RESULTS
Adding tier large
  IR001  IR003  IR009  IR027  IR081  IR243  Complete
Adding tier medium
  IR001  IR003  IR009  IR027  IR081  IR243  Complete
Adding tier small
  IR001  IR003  IR009  IR027  IR081  IR243  Complete
Adding tier medium
  IR001  IR003  IR009  IR027  IR081  Complete
Adding tier medium
  IR001  IR243  IR729  Complete
Adding tier small
  IR001  IR243  Complete
Adding tier smallaug
  IR001  IR003  IR009  IR027  IR081  Complete
Adding tier smalla
  IR001  IR003  IR009  Complete
Adding tier smallb
  IR001  IR003  IR009  Complete
Adding tier smallc
  IR001  IR003  IR009  Complete
Adding tier smalld
  IR001  IR003  IR009  Complete
Adding tier xsmalla
  IR001  IR003  IR009  IR027  Complete
Adding tier xsmallb
  IR001  IR003  IR009  IR027  Complete
Adding tier xsmallc
  IR001  IR003  IR009  IR027  Complete
Adding tier xsmalld
  IR001  IR003  IR009  IR027  Complete

SCENARIO LEVEL RESULTS
Adding tier large
  IR001  IR003  IR009  IR027  IR081  IR243  Complete
Addin

### Save results for Gaussian Naive Bayes

In [141]:
# Call the wrapper function for GNB
hf_save_f1_recovery_model(gnb_stages, 0, 'gnb')


FOLD LEVEL RESULTS
Adding tier large
  IR001  IR003  IR009  IR027  IR081  IR243  Complete
Adding tier medium
  IR001  IR003  IR009  IR027  IR081  Complete
Adding tier small
  IR001  IR003  IR009  IR027  IR081  Complete
Adding tier smallaug
  IR001  IR003  IR009  IR027  IR081  Complete
Adding tier medium
  IR001  IR003  IR009  IR027  IR081  Complete
Adding tier smalla
  IR001  IR003  IR009  Complete
Adding tier smallb
  IR001  IR003  IR009  Complete
Adding tier smallc
  IR001  IR003  IR009  Complete
Adding tier smalld
  IR001  IR003  IR009  Complete
Adding tier xsmalla
  IR001  IR003  Complete
Adding tier xsmallb
  IR001  IR003  Complete
Adding tier xsmallc
  IR001  IR003  Complete
Adding tier xsmalld
  IR001  IR003  Complete
Adding tier medium
  IR001  IR243  IR729  Complete
Adding tier small
  IR001  IR243  Complete

SCENARIO LEVEL RESULTS
Adding tier large
  IR001  IR003  IR009  IR027  IR081  IR243  Complete
Adding tier medium
  IR001  IR003  IR009  IR027  IR081  Complete
Adding tie

*<center>-- End of notebook --</center>*